## Which of the 25 countries has sufficent data quaility and temporal coverage to remain in the running for thee modeling dataset

In [23]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [24]:
analysis_df = pd.read_csv("../data/processed/analysis_df.csv")

print(analysis_df.shape)
analysis_df.head()

(175, 17)


,AreaName,AreaCode,Year,Active mobile-broadband subscriptions,Estimated proportion of households with Internet access at home,Fixed-broadband subscriptions,Internet users (%),Percentage of the population covered by at least an LTE/WiMAX mobile network.,Population,Country Name,Country Code,Electricity_Access,Region,Income group,Mobile_Broadband_per_100,Fixed-broadband_per_100,Coverage_Use_Gap
0,Afghanistan,AFG,2018,6997208.0,7.270304,15999.0,16.799999,7.00,36743039.0,Afghanistan,AFG,93.4,"Middle East, North Africa, Afghanistan & Pakistan",Low income,19.043629,0.043543,-9.799999
1,Afghanistan,AFG,2019,7309797.0,8.438115,19683.0,17.600000,22.00,37856121.0,Afghanistan,AFG,97.7,"Middle East, North Africa, Afghanistan & Pakistan",Low income,19.309419,0.051994,4.400000
2,Afghanistan,AFG,2020,7422550.0,NaN,26570.0,17.048500,26.00,39068979.0,Afghanistan,AFG,97.7,"Middle East, North Africa, Afghanistan & Pakistan",Low income,18.998577,0.068008,8.951500
3,Afghanistan,AFG,2021,7422550.0,24.288903,26570.0,16.514299,26.00,40000412.0,Afghanistan,AFG,97.7,"Middle East, North Africa, Afghanistan & Pakistan",Low income,18.556184,0.066424,9.485701
4,Afghanistan,AFG,2022,22831896.0,27.592028,32313.0,15.866300,33.03,40578842.0,Afghanistan,AFG,85.3,"Middle East, North Africa, Afghanistan & Pakistan",Low income,56.265519,0.079630,17.163700


In [25]:
# Creating table to establish which countries have the most useful data for training the model

model_vars = [
    "Electricity_Access",
    "Internet users (%)",
    "Mobile_Broadband_per_100",
    "Fixed-broadband_per_100",
    "Percentage of the population covered by at least an LTE/WiMAX mobile network."
]

country_data_completeness = (
    analysis_df
    .groupby("Country Name")[model_vars]
    .agg(["count",( "missing", lambda x: x.isna().sum())])
)

country_data_completeness

Electricity_Access         Internet users (%)          \
                              count missing              count missing   
Country Name                                                             
Afghanistan                       7       0                  7       0   
Angola                            7       0                  7       0   
Armenia                           7       0                  7       0   
Barbados                          7       0                  7       0   
Botswana                          7       0                  6       1   
Burkina Faso                      7       0                  7       0   
Cambodia                          7       0                  7       0   
Colombia                          7       0                  7       0   
Ethiopia                          7       0                  7       0   
Honduras                          7       0                  7       0   
India                             7       0                  7       0   
Indonesia                         7       0                  7       0   
Iraq                              7       0                  7       0   
Israel                            7       0                  7       0   
Jamaica                           7       0                  7       0   
Japan                             7       0                  7       0   
Kenya                             7       0                  7       0   
Madagascar                        7       0                  7       0   
Morocco                           7       0                  7       0   
Papua New Guinea                  7       0                  7       0   
Rwanda                            7       0                  7       0   
Spain                             7       0                  7       0   
Uganda                            7       0                  7       0   
Ukraine                           7       0                  7       0   
United States                     7       0                  7       0   

                 Mobile_Broadband_per_100         Fixed-broadband_per_100  \
                                    count missing                   count   
Country Name                                                                
Afghanistan                             7       0                       6   
Angola                                  7       0                       7   
Armenia                                 7       0                       7   
Barbados                                6       1                       5   
Botswana                                7       0                       7   
Burkina Faso                            5       2                       7   
Cambodia                                7       0                       7   
Colombia                                7       0                       7   
Ethiopia                                6       1                       6   
Honduras                                7       0                       7   
India                                   7       0                       7   
Indonesia                               7       0                       7   
Iraq                                    7       0                       7   
Israel                                  7       0                       7   
Jamaica                                 7       0                       7   
Japan                                   6       1                       6   
Kenya                                   7       0                       7   
Madagascar                              7       0                       6   
Morocco                                 7       0                       7   
Papua New Guinea                        5       2                       6   
Rwanda                                  7       0                       7   
Spain                                   7       0                       7   
Uganda             

In [26]:
# Locating the missing values in data

missing_records = (
    analysis_df[analysis_df[model_vars].isna().any(axis=1)]
    [["Country Name", "Year"] + model_vars].sort_values(['Country Name', "Year"]) 
)

missing_records

,Country Name,Year,Electricity_Access,Internet users (%),Mobile_Broadband_per_100,Fixed-broadband_per_100,Percentage of the population covered by at least an LTE/WiMAX mobile network.
6,Afghanistan,2024,87.8,16.094999,60.099082,NaN,37.324101
26,Barbados,2023,100.0,69.675102,NaN,NaN,NaN
27,Barbados,2024,100.0,70.364700,59.936205,NaN,99.000000
33,Botswana,2023,76.0,NaN,111.872300,6.164958,91.000000
37,Burkina Faso,2020,18.5,12.973200,NaN,0.065083,26.920000
40,Burkina Faso,2023,21.7,25.361200,NaN,0.459463,42.390000
56,Ethiopia,2018,44.8,15.886200,NaN,NaN,NaN
111,Japan,2024,100.0,85.541531,NaN,NaN,98.100000
125,Madagascar,2024,42.5,18.677900,28.197129,NaN,34.626694
133,Papua New Guinea,2018,16.6,12.357900,NaN,NaN,NaN


## Modeling Variable Selection

Internet use (%) was selected as the initial target variable because the project seeks to examine how differences in digital infrastructure relate to actual digital adoption. Candidate predictors include electricity access, LTE/WiMAX mobile network coverage, mobile-broadband subscriptions per 100 people, and fixed-broadband subscriptions per 100 people.

Income group and year will be evaluated as additional contextual and temporal predictors. Country name will be retained as an identifier rather than treated as a numeric predictor.

Coverage_Use_Gap will not be used as a predictor of Internet use because the variable is calculated using Internet use itself. Including it would introduce target leakage.
(Coverage_Use_Gap = LTECoverage - InternetUse)

In [27]:
## Creating modeling target variables

target = "Internet users (%)"

predictors = [
    "Electricity_Access",
    "Mobile_Broadband_per_100",
    "Fixed-broadband_per_100",
    "Percentage of the population covered by at least an LTE/WiMAX mobile network."
]

In [28]:
## Creating modeling dataframe with only the countries that have complete data for all model variables

modeling_columns = [
    "Country Name",
    "Year",
    "Income group",
    "Region",
    target
] + predictors 

modeling_df = analysis_df[modeling_columns].copy()

print(modeling_df.shape)
modeling_df.head()


(175, 9)


,Country Name,Year,Income group,Region,Internet users (%),Electricity_Access,Mobile_Broadband_per_100,Fixed-broadband_per_100,Percentage of the population covered by at least an LTE/WiMAX mobile network.
0,Afghanistan,2018,Low income,"Middle East, North Africa, Afghanistan & Pakistan",16.799999,93.4,19.043629,0.043543,7.00
1,Afghanistan,2019,Low income,"Middle East, North Africa, Afghanistan & Pakistan",17.600000,97.7,19.309419,0.051994,22.00
2,Afghanistan,2020,Low income,"Middle East, North Africa, Afghanistan & Pakistan",17.048500,97.7,18.998577,0.068008,26.00
3,Afghanistan,2021,Low income,"Middle East, North Africa, Afghanistan & Pakistan",16.514299,97.7,18.556184,0.066424,26.00
4,Afghanistan,2022,Low income,"Middle East, North Africa, Afghanistan & Pakistan",15.866300,85.3,56.265519,0.079630,33.03


In [29]:
modeling_df.dtypes

Country Name                                                                      object
Year                                                                               int64
Income group                                                                      object
Region                                                                            object
Internet users (%)                                                               float64
Electricity_Access                                                               float64
Mobile_Broadband_per_100                                                         float64
Fixed-broadband_per_100                                                          float64
Percentage of the population covered by at least an LTE/WiMAX mobile network.    float64
dtype: object

## Income Group Selection Rationale

Income group will initially be excluded from the predictor variables and retained as a segmentation variable for model evaluation. This approach allows the initial models to assess the predictive value of digital infrastructure indicators independently of income classification. Model performance will subsequently be compared across income groups to identify potential differences in predictive accuracy. Income group may be introduced as an additional predictor in later model comparisons to determine whether economic classification provides incremental predictive value beyond the infrastructure indicators.

## Region Selection Rationale
Region will be excluded from the initial predictor variables due to the limited number of countries represented in the dataset and potential imbalances in geographic representation. Including region at this stage could introduce additional model complexity and increase the risk of overfitting to country-specific characteristics rather than identifying generalizable relationships. Region will be retained for descriptive analysis and may be reconsidered as a predictor in future modeling iterations if geographic representation and sample size improve.

## Temporal Variable Selection Rationale
Year will be included as an initial predictor to account for temporal patterns in internet adoption between 2018 and 2024. Including this variable may help identify changes in internet use that are not fully captured by the selected infrastructure indicators. However, Year will not be interpreted as a causal explanation for changes in adoption. Its predictive contribution will be evaluated using a chronological validation strategy to determine whether including temporal information improves model performance on later observations.

In [30]:
# Checking for missing values in the modeling dataframe

modeling_df.isna().sum()

Country Name                                                                     0
Year                                                                             0
Income group                                                                     0
Region                                                                           0
Internet users (%)                                                               1
Electricity_Access                                                               0
Mobile_Broadband_per_100                                                         7
Fixed-broadband_per_100                                                          8
Percentage of the population covered by at least an LTE/WiMAX mobile network.    5
dtype: int64

In [31]:
modeling_df[modeling_df["Internet users (%)"].isna()]

,Country Name,Year,Income group,Region,Internet users (%),Electricity_Access,Mobile_Broadband_per_100,Fixed-broadband_per_100,Percentage of the population covered by at least an LTE/WiMAX mobile network.
33,Botswana,2023,Upper middle income,Sub-Saharan Africa,NaN,76.0,111.8723,6.164958,91.0


In [ ]:
# Receieved a type error that modeling_df was a list so I had to restore it to a dataframe. 
modeling_df = analysis_df[modeling_columns].copy()

In [32]:
modeling_df[target].isna().sum()

1

In [34]:
# Keeping only the rows with complete data for all model variables

supervised_modeling_df = modeling_df.dropna(subset=[target]).copy()

print('Original observations:', len(modeling_df))
print('Supervised observations:', len(supervised_modeling_df))

Original observations: 175
Supervised observations: 174


In [36]:
# Looking to see how many predictor values are missing in each year for the supervised modeling dataframe

supervised_modeling_df.groupby('Year')[predictors].apply(lambda df: df.isna().sum())

,Electricity_Access,Mobile_Broadband_per_100,Fixed-broadband_per_100,Percentage of the population covered by at least an LTE/WiMAX mobile network.
Year,,,,
2018,0,2,3,4
2019,0,0,0,0
2020,0,1,0,0
2021,0,0,0,0
2022,0,0,0,0
2023,0,3,1,1
2024,0,1,4,0
